In [ ]:
from pathlib import Path
import html
import importlib.util
import os
import random
import re
import shutil
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    required_packages = [
        "beautifulsoup4",
        "tqdm",
        "transformers",
        "datasets",
        "accelerate",
        "joblib",
        "seaborn",
        "scikit-learn",
        "nltk",
    ]
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *required_packages,
    ])

# Keep Hugging Face downloads inside the project/session instead of the user cache.
_initial_dir = Path.cwd()
_initial_project_root = Path("/content") if IN_COLAB else (
    _initial_dir.parent if _initial_dir.name == "Model Trained" else _initial_dir
)
_initial_hf_cache = _initial_project_root / "hf_cache"
os.environ.setdefault("HF_HOME", str(_initial_hf_cache))
os.environ.setdefault("TRANSFORMERS_CACHE", str(_initial_hf_cache / "transformers"))

import joblib
import nltk
import numpy as np
import pandas as pd
import torch
from bs4 import BeautifulSoup
from datasets import Dataset
from nltk import pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

tqdm.pandas()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Running in Colab:", IN_COLAB)
print("GPU available:", torch.cuda.is_available())

In [ ]:
# Resolve paths so the notebook works locally or in Google Colab.
current_dir = Path.cwd()
PROJECT_ROOT = Path("/content") if IN_COLAB else (
    current_dir.parent if current_dir.name == "Model Trained" else current_dir
)

DATA_DIR = PROJECT_ROOT / "dataset"
DATA_DIR.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    DATA_DIR / "emails.csv",
    PROJECT_ROOT / "emails.csv",
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None and IN_COLAB:
    from google.colab import files

    print("Upload the new emails.csv dataset when prompted.")
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError("No CSV file was uploaded. Please upload emails.csv.")

    uploaded_path = PROJECT_ROOT / csv_files[0]
    target_path = DATA_DIR / "emails.csv"
    shutil.copy2(uploaded_path, target_path)
    DATA_PATH = target_path

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find dataset/emails.csv. Put emails.csv in the dataset folder "
        "or upload it when running in Colab."
    )

PREPROCESSED_PATH = DATA_DIR / "final_preprocessed_emails_dataset.csv"
HF_CACHE_DIR = PROJECT_ROOT / "hf_cache"
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR / "transformers")
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

APP_MODEL_DIR = PROJECT_ROOT / "spam_email_detector" / "models"
NOTEBOOK_MODEL_DIR = PROJECT_ROOT / "Model Trained" / "saved_models"

APP_MODEL_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Models will be saved to:", APP_MODEL_DIR)

In [ ]:
NLTK_DATA_DIR = PROJECT_ROOT / "nltk_data"
NLTK_DATA_DIR.mkdir(parents=True, exist_ok=True)
if str(NLTK_DATA_DIR) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_DATA_DIR))

required_nltk_resources = {
    "tokenizers/punkt": "punkt",
    "tokenizers/punkt_tab": "punkt_tab",
    "corpora/stopwords": "stopwords",
    "corpora/wordnet": "wordnet",
    "corpora/omw-1.4": "omw-1.4",
    "taggers/averaged_perceptron_tagger_eng": "averaged_perceptron_tagger_eng",
}


def nltk_resource_exists(resource_path):
    try:
        nltk.data.find(resource_path)
        return True
    except LookupError:
        try:
            nltk.data.find(resource_path + ".zip")
            return True
        except LookupError:
            return False


for resource_path, download_name in required_nltk_resources.items():
    if not nltk_resource_exists(resource_path):
        nltk.download(download_name, download_dir=str(NLTK_DATA_DIR), quiet=True)

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing values:")
print(df.isna().sum())
print("Class distribution:")
print(df["spam"].value_counts().sort_index())

df.head()

In [ ]:
# New dataset format: text contains the full email and spam is 0 for ham, 1 for spam.
required_columns = {"text", "spam"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = df[["text", "spam"]].copy()
df["raw_text"] = df["text"].fillna("").astype(str)
df["label_num"] = pd.to_numeric(df["spam"], errors="coerce")

df = df[
    (df["raw_text"].str.strip() != "")
    & (df["label_num"].isin([0, 1]))
].copy()

df["label_num"] = df["label_num"].astype(int)
df["label"] = df["label_num"].map({0: "ham", 1: "spam"})

# Use blank subject/message columns to preserve the original notebook output schema.
df["subject"] = ""
df["message"] = df["raw_text"]

# Remove exact duplicate emails with the same label.
df = df.drop_duplicates(subset=["raw_text", "label_num"]).reset_index(drop=True)

print("Shape after initial cleaning:", df.shape)
print("Class distribution:")
print(df["label"].value_counts())

In [ ]:
lemmatizer = WordNetLemmatizer()

# Remove common stopwords but preserve negation words,
# because words such as "not" can affect meaning.
stop_words = set(stopwords.words("english")) - {"no", "nor", "not"}


def get_wordnet_pos(tag):
    """Convert NLTK POS tags to WordNet tags."""
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN


def preprocess_text(text):
    text = html.unescape(str(text))
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    tokens = [
        token for token in tokens
        if token not in stop_words and len(token) > 1
    ]

    tagged_tokens = pos_tag(tokens)
    tokens = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged_tokens
    ]

    return " ".join(tokens)

In [ ]:
df["cleaned_text"] = df["raw_text"].progress_apply(preprocess_text)

# Remove rows that became empty.
df = df[df["cleaned_text"].str.strip() != ""].copy()

# Remove ambiguous cleaned texts that have conflicting labels.
conflicting_texts = df.groupby("cleaned_text")["label_num"].nunique()
conflicting_texts = conflicting_texts[conflicting_texts > 1].index

df = df[~df["cleaned_text"].isin(conflicting_texts)].copy()
df = df.drop_duplicates(subset=["cleaned_text", "label_num"]).reset_index(drop=True)

df["word_count"] = df["cleaned_text"].str.split().str.len()

final_df = df[
    [
        "subject",
        "message",
        "raw_text",
        "label",
        "cleaned_text",
        "word_count",
        "label_num",
    ]
]

final_df.to_csv(PREPROCESSED_PATH, index=False)

print("Saved:", PREPROCESSED_PATH)
print("Final shape:", final_df.shape)
print("Class distribution:")
print(final_df["label"].value_counts())

final_df.head()

In [ ]:
X = final_df["cleaned_text"]
y = final_df["label_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training class distribution:")
print(y_train.value_counts())

In [ ]:
bow_vectorizer = CountVectorizer(
    max_features=10000,
    min_df=2,
    max_df=0.95,
)

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

print("BoW training shape:", X_train_bow.shape)
print("BoW testing shape:", X_test_bow.shape)
print("Sample features:", bow_vectorizer.get_feature_names_out()[:20])

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=15000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape:", X_test_tfidf.shape)
print("Sample features:", tfidf_vectorizer.get_feature_names_out()[:20])

In [ ]:
train_indices, test_indices = train_test_split(
    np.arange(len(final_df)),
    test_size=0.20,
    random_state=SEED,
    stratify=final_df["label_num"],
)

train_df = final_df.iloc[train_indices].reset_index(drop=True)
test_df = final_df.iloc[test_indices].reset_index(drop=True)

# BERT uses the natural, unprocessed email text.
train_df["bert_text"] = train_df["raw_text"]
test_df["bert_text"] = test_df["raw_text"]

y_train = train_df["label_num"]
y_test = test_df["label_num"]

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df["cleaned_text"])
X_test_tfidf = tfidf_vectorizer.transform(test_df["cleaned_text"])

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))
print("Training labels:")
print(train_df["label"].value_counts())
print("Testing labels:")
print(test_df["label"].value_counts())
print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)

In [ ]:
results = []
confusion_matrices = {}


def evaluate_model(model_name, y_true, y_pred):
    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

    results.append(metrics)
    confusion_matrices[model_name] = confusion_matrix(y_true, y_pred)

    print(f"\n{model_name}")
    print("-" * 50)
    print(classification_report(
        y_true,
        y_pred,
        target_names=["Ham", "Spam"],
        zero_division=0,
    ))

    return metrics

In [ ]:
nb_model = MultinomialNB(alpha=0.5)
nb_model.fit(X_train_tfidf, y_train)
nb_predictions = nb_model.predict(X_test_tfidf)

evaluate_model(
    "Multinomial Naive Bayes",
    y_test,
    nb_predictions,
)

In [ ]:
lr_model = LogisticRegression(
    C=4.0,
    max_iter=2000,
    class_weight="balanced",
    random_state=SEED,
)

lr_model.fit(X_train_tfidf, y_train)
lr_predictions = lr_model.predict(X_test_tfidf)

evaluate_model(
    "Logistic Regression",
    y_test,
    lr_predictions,
)

In [ ]:
BERT_MODEL_NAME = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

In [ ]:
bert_train_dataset = Dataset.from_dict({
    "text": train_df["bert_text"].tolist(),
    "labels": train_df["label_num"].tolist(),
})

bert_test_dataset = Dataset.from_dict({
    "text": test_df["bert_text"].tolist(),
    "labels": test_df["label_num"].tolist(),
})


def tokenize_emails(batch):
    return bert_tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )


bert_train_dataset = bert_train_dataset.map(
    tokenize_emails,
    batched=True,
    remove_columns=["text"],
)

bert_test_dataset = bert_test_dataset.map(
    tokenize_emails,
    batched=True,
    remove_columns=["text"],
)

bert_train_dataset

In [ ]:
def compute_bert_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, pos_label=1, zero_division=0),
        "recall": recall_score(labels, predictions, pos_label=1, zero_division=0),
        "f1": f1_score(labels, predictions, pos_label=1, zero_division=0),
    }

In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=2,
    id2label={0: "HAM", 1: "SPAM"},
    label2id={"HAM": 0, "SPAM": 1},
)

training_arguments = TrainingArguments(
    output_dir=str(PROJECT_ROOT / "bert_training_output"),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

bert_trainer = Trainer(
    model=bert_model,
    args=training_arguments,
    train_dataset=bert_train_dataset,
    eval_dataset=bert_test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_bert_metrics,
)

bert_trainer.train()

In [ ]:
bert_output = bert_trainer.predict(bert_test_dataset)
bert_logits = bert_output.predictions
bert_predictions = np.argmax(bert_logits, axis=1)

evaluate_model(
    "BERT",
    y_test.to_numpy(),
    bert_predictions,
)

english_bert_confusion_df = pd.DataFrame(
    confusion_matrices["BERT"],
    index=["Actual Ham", "Actual Spam"],
    columns=["Predicted Ham", "Predicted Spam"],
)
english_bert_confusion_df

In [ ]:
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(
    by="F1-Score",
    ascending=False,
).reset_index(drop=True)

for column in ["Accuracy", "Precision", "Recall", "F1-Score"]:
    comparison_df[column] = comparison_df[column].round(4)

comparison_df

In [ ]:
# Save traditional models and BERT using the same filenames expected by the app.
joblib.dump(tfidf_vectorizer, APP_MODEL_DIR / "tfidf_vectorizer.joblib")
joblib.dump(nb_model, APP_MODEL_DIR / "naive_bayes_model.joblib")
joblib.dump(lr_model, APP_MODEL_DIR / "logistic_regression_model.joblib")

shutil.copy2(APP_MODEL_DIR / "tfidf_vectorizer.joblib", NOTEBOOK_MODEL_DIR / "tfidf_vectorizer.joblib")
shutil.copy2(APP_MODEL_DIR / "naive_bayes_model.joblib", NOTEBOOK_MODEL_DIR / "naive_bayes_model.joblib")
shutil.copy2(APP_MODEL_DIR / "logistic_regression_model.joblib", NOTEBOOK_MODEL_DIR / "logistic_regression_model.joblib")

BERT_APP_DIR = APP_MODEL_DIR / "bert_spam_model"
BERT_NOTEBOOK_DIR = NOTEBOOK_MODEL_DIR / "bert_spam_model"

bert_trainer.save_model(BERT_APP_DIR)
bert_tokenizer.save_pretrained(BERT_APP_DIR)

if BERT_NOTEBOOK_DIR.exists():
    shutil.rmtree(BERT_NOTEBOOK_DIR)
shutil.copytree(BERT_APP_DIR, BERT_NOTEBOOK_DIR)

comparison_df.to_csv(APP_MODEL_DIR / "model_comparison.csv", index=False)
shutil.copy2(APP_MODEL_DIR / "model_comparison.csv", NOTEBOOK_MODEL_DIR / "model_comparison.csv")

best_model_name = comparison_df.iloc[0]["Model"]
best_f1 = comparison_df.iloc[0]["F1-Score"]

print("Saved traditional models and BERT to:", APP_MODEL_DIR)
print("Best model:", best_model_name)
print("Best spam F1-score:", best_f1)

In [ ]:
test_tokenizer = AutoTokenizer.from_pretrained(BERT_APP_DIR)
test_model = AutoModelForSequenceClassification.from_pretrained(BERT_APP_DIR)
test_model.eval()

sample_email = """
Congratulations! You have won a free cash prize.
Click here now to claim your reward.
"""

inputs = test_tokenizer(
    sample_email,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256,
)

with torch.no_grad():
    logits = test_model(**inputs).logits
    probabilities = torch.softmax(logits, dim=1)[0]

prediction = torch.argmax(probabilities).item()

print("Prediction:", "SPAM" if prediction == 1 else "HAM")
print("Ham probability:", probabilities[0].item())
print("Spam probability:", probabilities[1].item())

## Malay Spam/Phishing Model

This section trains a separate Malay model from `train_data_malay.csv` and `test_data_malay.csv`. It does not replace the English model artifacts.

In [ ]:
MALAY_TRAIN_PATH = DATA_DIR / "train_data_malay.csv"
MALAY_TEST_PATH = DATA_DIR / "test_data_malay.csv"

if IN_COLAB and (not MALAY_TRAIN_PATH.exists() or not MALAY_TEST_PATH.exists()):
    from google.colab import files

    print("Upload train_data_malay.csv and test_data_malay.csv when prompted.")
    uploaded = files.upload()
    for uploaded_name in uploaded:
        if uploaded_name in {"train_data_malay.csv", "test_data_malay.csv"}:
            shutil.copy2(PROJECT_ROOT / uploaded_name, DATA_DIR / uploaded_name)

if not MALAY_TRAIN_PATH.exists() or not MALAY_TEST_PATH.exists():
    raise FileNotFoundError(
        "Malay dataset files are required: dataset/train_data_malay.csv "
        "and dataset/test_data_malay.csv"
    )

malay_train_df = pd.read_csv(MALAY_TRAIN_PATH)
malay_test_df = pd.read_csv(MALAY_TEST_PATH)

print("Malay train shape:", malay_train_df.shape)
print("Malay test shape:", malay_test_df.shape)
print("Malay train columns:", malay_train_df.columns.tolist())
print("Malay train label distribution:")
print(malay_train_df["label"].value_counts())
print("Malay test label distribution:")
print(malay_test_df["label"].value_counts())

In [ ]:
malay_label_map = {
    "legit": 0,
    "ham": 0,
    "phishing": 1,
    "spam": 1,
}


def preprocess_malay_text(text):
    text = html.unescape(str(text))
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " url ", text)
    text = re.sub(r"\S+@\S+", " email ", text)
    text = re.sub(r"\d+", " number ", text)
    text = re.sub(r"[^a-zA-ZÀ-ÿ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return " ".join(token for token in text.split() if len(token) > 1)


def prepare_malay_dataframe(dataframe):
    missing_columns = {"text", "label"} - set(dataframe.columns)
    if missing_columns:
        raise ValueError(f"Malay dataset missing columns: {sorted(missing_columns)}")

    prepared = dataframe[["text", "label"]].copy()
    prepared["raw_text"] = prepared["text"].fillna("").astype(str)
    prepared["label"] = prepared["label"].fillna("").str.lower().str.strip()
    prepared["label_num"] = prepared["label"].map(malay_label_map)
    prepared["cleaned_text"] = prepared["raw_text"].progress_apply(preprocess_malay_text)

    prepared = prepared[
        (prepared["raw_text"].str.strip() != "")
        & (prepared["cleaned_text"].str.strip() != "")
        & (prepared["label_num"].notna())
    ].copy()
    prepared["label_num"] = prepared["label_num"].astype(int)
    return prepared.reset_index(drop=True)


malay_train_df = prepare_malay_dataframe(malay_train_df)
malay_test_df = prepare_malay_dataframe(malay_test_df)

MALAY_PREPROCESSED_TRAIN_PATH = DATA_DIR / "final_preprocessed_malay_train_dataset.csv"
MALAY_PREPROCESSED_TEST_PATH = DATA_DIR / "final_preprocessed_malay_test_dataset.csv"

malay_train_df[["raw_text", "label", "cleaned_text", "label_num"]].to_csv(
    MALAY_PREPROCESSED_TRAIN_PATH,
    index=False,
)
malay_test_df[["raw_text", "label", "cleaned_text", "label_num"]].to_csv(
    MALAY_PREPROCESSED_TEST_PATH,
    index=False,
)

print("Prepared Malay train shape:", malay_train_df.shape)
print("Prepared Malay test shape:", malay_test_df.shape)

In [ ]:
malay_tfidf_vectorizer = TfidfVectorizer(
    max_features=12000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

X_malay_train_tfidf = malay_tfidf_vectorizer.fit_transform(
    malay_train_df["cleaned_text"]
)
X_malay_test_tfidf = malay_tfidf_vectorizer.transform(
    malay_test_df["cleaned_text"]
)
y_malay_train = malay_train_df["label_num"]
y_malay_test = malay_test_df["label_num"]

print("Malay training matrix:", X_malay_train_tfidf.shape)
print("Malay testing matrix:", X_malay_test_tfidf.shape)

In [ ]:
malay_results = []
malay_confusion_matrices = {}


def evaluate_malay_model(model_name, y_true, y_pred):
    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

    malay_results.append(metrics)
    malay_confusion_matrices[model_name] = confusion_matrix(y_true, y_pred)

    print(f"\n{model_name}")
    print("-" * 50)
    print(classification_report(
        y_true,
        y_pred,
        target_names=["Legit", "Phishing"],
        zero_division=0,
    ))

    return metrics


malay_nb_model = MultinomialNB(alpha=0.5)
malay_nb_model.fit(X_malay_train_tfidf, y_malay_train)
malay_nb_predictions = malay_nb_model.predict(X_malay_test_tfidf)
evaluate_malay_model("Malay Multinomial Naive Bayes", y_malay_test, malay_nb_predictions)

malay_lr_model = LogisticRegression(
    C=4.0,
    max_iter=2000,
    class_weight="balanced",
    random_state=SEED,
)
malay_lr_model.fit(X_malay_train_tfidf, y_malay_train)
malay_lr_predictions = malay_lr_model.predict(X_malay_test_tfidf)
evaluate_malay_model("Malay Logistic Regression", y_malay_test, malay_lr_predictions)

In [ ]:
MALAY_BERT_MODEL_NAME = "bert-base-multilingual-cased"

malay_bert_tokenizer = AutoTokenizer.from_pretrained(MALAY_BERT_MODEL_NAME)
malay_data_collator = DataCollatorWithPadding(tokenizer=malay_bert_tokenizer)

malay_bert_train_dataset = Dataset.from_dict({
    "text": malay_train_df["raw_text"].tolist(),
    "labels": malay_train_df["label_num"].tolist(),
})

malay_bert_test_dataset = Dataset.from_dict({
    "text": malay_test_df["raw_text"].tolist(),
    "labels": malay_test_df["label_num"].tolist(),
})


def tokenize_malay_emails(batch):
    return malay_bert_tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )


malay_bert_train_dataset = malay_bert_train_dataset.map(
    tokenize_malay_emails,
    batched=True,
    remove_columns=["text"],
)

malay_bert_test_dataset = malay_bert_test_dataset.map(
    tokenize_malay_emails,
    batched=True,
    remove_columns=["text"],
)

In [ ]:
def compute_malay_bert_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, pos_label=1, zero_division=0),
        "recall": recall_score(labels, predictions, pos_label=1, zero_division=0),
        "f1": f1_score(labels, predictions, pos_label=1, zero_division=0),
    }


malay_bert_model = AutoModelForSequenceClassification.from_pretrained(
    MALAY_BERT_MODEL_NAME,
    num_labels=2,
    id2label={0: "LEGIT", 1: "PHISHING"},
    label2id={"LEGIT": 0, "PHISHING": 1},
)

malay_training_arguments = TrainingArguments(
    output_dir=str(PROJECT_ROOT / "malay_bert_training_output"),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

malay_bert_trainer = Trainer(
    model=malay_bert_model,
    args=malay_training_arguments,
    train_dataset=malay_bert_train_dataset,
    eval_dataset=malay_bert_test_dataset,
    data_collator=malay_data_collator,
    compute_metrics=compute_malay_bert_metrics,
)

malay_bert_trainer.train()

In [ ]:
malay_bert_output = malay_bert_trainer.predict(malay_bert_test_dataset)
malay_bert_logits = malay_bert_output.predictions
malay_bert_predictions = np.argmax(malay_bert_logits, axis=1)

evaluate_malay_model(
    "Malay BERT",
    y_malay_test.to_numpy(),
    malay_bert_predictions,
)

malay_bert_confusion_df = pd.DataFrame(
    malay_confusion_matrices["Malay BERT"],
    index=["Actual Legit", "Actual Phishing"],
    columns=["Predicted Legit", "Predicted Phishing"],
)
display(malay_bert_confusion_df)

malay_comparison_df = pd.DataFrame(malay_results).sort_values(
    by="F1-Score",
    ascending=False,
).reset_index(drop=True)

for column in ["Accuracy", "Precision", "Recall", "F1-Score"]:
    malay_comparison_df[column] = malay_comparison_df[column].round(4)

malay_comparison_df

In [ ]:
joblib.dump(malay_tfidf_vectorizer, APP_MODEL_DIR / "malay_tfidf_vectorizer.joblib")
joblib.dump(malay_nb_model, APP_MODEL_DIR / "malay_naive_bayes_model.joblib")
joblib.dump(malay_lr_model, APP_MODEL_DIR / "malay_logistic_regression_model.joblib")

MALAY_BERT_APP_DIR = APP_MODEL_DIR / "malay_bert_spam_model"
MALAY_BERT_NOTEBOOK_DIR = NOTEBOOK_MODEL_DIR / "malay_bert_spam_model"

malay_bert_trainer.save_model(MALAY_BERT_APP_DIR)
malay_bert_tokenizer.save_pretrained(MALAY_BERT_APP_DIR)

if MALAY_BERT_NOTEBOOK_DIR.exists():
    shutil.rmtree(MALAY_BERT_NOTEBOOK_DIR)
shutil.copytree(MALAY_BERT_APP_DIR, MALAY_BERT_NOTEBOOK_DIR)

malay_comparison_df.to_csv(APP_MODEL_DIR / "malay_model_comparison.csv", index=False)
shutil.copy2(APP_MODEL_DIR / "malay_model_comparison.csv", NOTEBOOK_MODEL_DIR / "malay_model_comparison.csv")

print("Saved Malay traditional models and Malay BERT to:", APP_MODEL_DIR)
print("Best Malay model:", malay_comparison_df.iloc[0]["Model"])
print("Best Malay phishing F1-score:", malay_comparison_df.iloc[0]["F1-Score"])

In [ ]:
if IN_COLAB:
    from google.colab import files

    english_zip_base = PROJECT_ROOT / "bert_spam_model"
    english_zip_path = shutil.make_archive(
        str(english_zip_base),
        "zip",
        root_dir=APP_MODEL_DIR,
        base_dir="bert_spam_model",
    )
    files.download(english_zip_path)

    malay_bert_dir = APP_MODEL_DIR / "malay_bert_spam_model"
    if malay_bert_dir.exists():
        malay_zip_base = PROJECT_ROOT / "malay_bert_spam_model"
        malay_zip_path = shutil.make_archive(
            str(malay_zip_base),
            "zip",
            root_dir=APP_MODEL_DIR,
            base_dir="malay_bert_spam_model",
        )
        files.download(malay_zip_path)

    classic_artifacts = [
        "malay_tfidf_vectorizer.joblib",
        "malay_naive_bayes_model.joblib",
        "malay_logistic_regression_model.joblib",
    ]
    existing_classic_artifacts = [
        filename for filename in classic_artifacts
        if (APP_MODEL_DIR / filename).exists()
    ]
    if existing_classic_artifacts:
        malay_classic_zip_base = PROJECT_ROOT / "malay_classic_models"
        temp_classic_dir = PROJECT_ROOT / "malay_classic_models"
        temp_classic_dir.mkdir(exist_ok=True)
        for filename in existing_classic_artifacts:
            shutil.copy2(APP_MODEL_DIR / filename, temp_classic_dir / filename)
        malay_classic_zip_path = shutil.make_archive(
            str(malay_classic_zip_base),
            "zip",
            root_dir=temp_classic_dir,
        )
        files.download(malay_classic_zip_path)

    for output_path in [
        APP_MODEL_DIR / "model_comparison.csv",
        APP_MODEL_DIR / "malay_model_comparison.csv",
        PREPROCESSED_PATH,
        DATA_DIR / "final_preprocessed_malay_train_dataset.csv",
        DATA_DIR / "final_preprocessed_malay_test_dataset.csv",
    ]:
        if output_path.exists():
            files.download(str(output_path))
else:
    print("English BERT model saved to:", APP_MODEL_DIR / "bert_spam_model")
    print("Malay BERT model saved to:", APP_MODEL_DIR / "malay_bert_spam_model")
    print("Malay classic models saved to:", APP_MODEL_DIR)
    print("English comparison saved to:", APP_MODEL_DIR / "model_comparison.csv")
    print("Malay comparison saved to:", APP_MODEL_DIR / "malay_model_comparison.csv")